## --- Analyse Expérimentale Huffman Dynamique ---

In [ ]:
import os
def show_ratio(txt_rel, bin_rel, label=None):
    txt_path = os.path.join("..", *txt_rel.split("/"))
    bin_path = os.path.join("..", *bin_rel.split("/"))
    txt_size = os.path.getsize(txt_path) if os.path.exists(txt_path) else 0
    bin_size = os.path.getsize(bin_path) if os.path.exists(bin_path) else 0
    if not txt_size or not bin_size:
        print(f"{label or txt_rel}: fichiers introuvables ou taille nulle")
        return
    ratio = bin_size / txt_size
    taux = 1 - ratio
    facteur = txt_size / bin_size
    name = label or os.path.basename(txt_path)
    print(f"{name}: {bin_size}/{txt_size} octets | taux: {taux:.2%} | facteur: {facteur:.2f}x")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def load_log_file(filename):
    """
    Charge un fichier de log (compression.txt ou decompression.txt)
    
    Format attendu : input_file;output_file;input_size;output_size;rate;time_ms
    """
    columns = ['input_file', 'output_file', 'input_size', 'output_size', 'rate', 'time_ms']
    df = pd.read_csv(filename, sep=';', names=columns, encoding='utf-8')
    return df

In [ ]:
def plot_time_vs_size():
    """
    Crée des graphiques montrant le temps en fonction de la taille des fichiers
    """
    # Charger les données
    df_compression = load_log_file('../compression.txt')
    df_decompression = load_log_file('../decompression.txt')
    
    # Créer une figure avec 2 sous-graphiques
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ========== GRAPHIQUE 1 : Compression ==========
    ax1 = axes[0]
    
    # Scatter plot
    ax1.scatter(df_compression['input_size'], df_compression['time_ms'], 
                alpha=0.6, s=50, color='blue', label='Points de données')
    
    # Ligne de tendance (régression linéaire)
    z = np.polyfit(df_compression['input_size'], df_compression['time_ms'], 1)
    p = np.poly1d(z)
    ax1.plot(df_compression['input_size'], p(df_compression['input_size']), 
             "r--", linewidth=2, label=f'Tendance: y={z[0]:.6f}x+{z[1]:.2f}')
    
    ax1.set_xlabel('Taille du fichier (octets)', fontsize=12)
    ax1.set_ylabel('Temps de compression (ms)', fontsize=12)
    ax1.set_title('Temps de compression vs Taille du fichier', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # ========== GRAPHIQUE 2 : Décompression ==========
    ax2 = axes[1]
    
    # Scatter plot
    ax2.scatter(df_decompression['input_size'], df_decompression['time_ms'], 
                alpha=0.6, s=50, color='green', label='Points de données')
    
    # Ligne de tendance
    z2 = np.polyfit(df_decompression['input_size'], df_decompression['time_ms'], 1)
    p2 = np.poly1d(z2)
    ax2.plot(df_decompression['input_size'], p2(df_decompression['input_size']), 
             "r--", linewidth=2, label=f'Tendance: y={z2[0]:.6f}x+{z2[1]:.2f}')
    
    ax2.set_xlabel('Taille du fichier compressé (octets)', fontsize=12)
    ax2.set_ylabel('Temps de décompression (ms)', fontsize=12)
    ax2.set_title('Temps de décompression vs Taille du fichier', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('temps_vs_taille.png', dpi=300, bbox_inches='tight')
    print("✓ Graphique sauvegardé : temps_vs_taille.png")
    plt.show()


def plot_comparison():
    """
    Compare compression et décompression sur le même graphique
    """
    # Charger les données
    df_comp = load_log_file('../compression.txt')
    df_decomp = load_log_file('../decompression.txt')
    
    plt.figure(figsize=(10, 6))
    
    # Compression
    plt.scatter(df_comp['input_size'], df_comp['time_ms'], 
                alpha=0.6, s=50, color='blue', label='Compression')
    
    # Décompression
    plt.scatter(df_decomp['input_size'], df_decomp['time_ms'], 
                alpha=0.6, s=50, color='green', label='Décompression')
    
    plt.xlabel('Taille du fichier (octets)', fontsize=12)
    plt.ylabel('Temps (ms)', fontsize=12)
    plt.title('Comparaison Compression vs Décompression', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('comparaison_temps.png', dpi=300, bbox_inches='tight')
    print("✓ Graphique sauvegardé : comparaison_temps.png")
    plt.show()


def plot_compression_rate():
    """
    Graphique du taux de compression en fonction de la taille
    """
    df = load_log_file('../compression.txt')
    
    plt.figure(figsize=(10, 6))
    
    plt.scatter(df['input_size'], df['rate'], alpha=0.6, s=50, color='purple')
    
    # Ligne horizontale à 1.0 (pas de compression)
    plt.axhline(y=1.0, color='r', linestyle='--', linewidth=1, label='Pas de compression (rate=1.0)')
    
    plt.xlabel('Taille du fichier original (octets)', fontsize=12)
    plt.ylabel('Taux de compression', fontsize=12)
    plt.title('Taux de compression vs Taille du fichier', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('taux_compression.png', dpi=300, bbox_inches='tight')
    print("✓ Graphique sauvegardé : taux_compression.png")
    plt.show()


def plot_all_stats():
    """
    Crée une figure avec 4 graphiques pour une vue complète
    """
    df_comp = load_log_file('../compression.txt')
    df_decomp = load_log_file('../decompression.txt')
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Temps compression vs taille
    axes[0, 0].scatter(df_comp['input_size'], df_comp['time_ms'], alpha=0.6, color='blue')
    axes[0, 0].set_xlabel('Taille (octets)')
    axes[0, 0].set_ylabel('Temps (ms)')
    axes[0, 0].set_title('Compression : Temps vs Taille')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Temps décompression vs taille
    axes[0, 1].scatter(df_decomp['input_size'], df_decomp['time_ms'], alpha=0.6, color='green')
    axes[0, 1].set_xlabel('Taille (octets)')
    axes[0, 1].set_ylabel('Temps (ms)')
    axes[0, 1].set_title('Décompression : Temps vs Taille')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Taux de compression vs taille
    axes[1, 0].scatter(df_comp['input_size'], df_comp['rate'], alpha=0.6, color='purple')
    axes[1, 0].axhline(y=1.0, color='r', linestyle='--', linewidth=1)
    axes[1, 0].set_xlabel('Taille (octets)')
    axes[1, 0].set_ylabel('Taux de compression')
    axes[1, 0].set_title('Taux de compression vs Taille')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Distribution des taux de compression
    axes[1, 1].hist(df_comp['rate'], bins=20, alpha=0.7, color='orange', edgecolor='black')
    axes[1, 1].axvline(x=df_comp['rate'].mean(), color='r', linestyle='--', 
                       linewidth=2, label=f'Moyenne: {df_comp["rate"].mean():.3f}')
    axes[1, 1].set_xlabel('Taux de compression')
    axes[1, 1].set_ylabel('Fréquence')
    axes[1, 1].set_title('Distribution des taux de compression')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('analyse_complete.png', dpi=300, bbox_inches='tight')
    print("✓ Graphique sauvegardé : analyse_complete.png")
    plt.show()


def print_statistics():
    """
    Affiche des statistiques résumées
    """
    df_comp = load_log_file('../compression.txt')
    df_decomp = load_log_file('../decompression.txt')
    
    print("=" * 70)
    print("STATISTIQUES DE COMPRESSION")
    print("=" * 70)
    print(f"Nombre de fichiers : {len(df_comp)}")
    print(f"Taux moyen : {df_comp['rate'].mean():.5f}")
    print(f"Temps moyen : {df_comp['time_ms'].mean():.2f} ms")
    print(f"Taille moyenne entrée : {df_comp['input_size'].mean():.0f} octets")
    print(f"Réduction moyenne : {(1 - df_comp['rate'].mean()) * 100:.2f}%")
    
    print("\n" + "=" * 70)
    print("STATISTIQUES DE DÉCOMPRESSION")
    print("=" * 70)
    print(f"Nombre de fichiers : {len(df_decomp)}")
    print(f"Temps moyen : {df_decomp['time_ms'].mean():.2f} ms")
    
    # Comparaison vitesse
    ratio = df_comp['time_ms'].mean() / df_decomp['time_ms'].mean()
    print(f"\n⚡ La décompression est {ratio:.2f}x plus rapide que la compression")


In [ ]:
# Afficher les statistiques
print_statistics()
print("\n")

# Générer les graphiques
print("Génération des graphiques...")

# Graphique 1 : Temps vs taille (compression et décompression séparés)
plot_time_vs_size()

# Graphique 2 : Comparaison directe
plot_comparison()

# Graphique 3 : Taux de compression
plot_compression_rate()

# Graphique 4 : Vue d'ensemble
plot_all_stats()


### Résultats expérimentaux sur fichiers textuels existants (question 8) 
Bibliothèques de textes utilisée : projet Gutenberg, cf rapport pour sources

#### Texte Français (NdP - Hugo)

In [ ]:
# Français (NdP - Hugo)
show_ratio("texts/francais.txt", "texts/bintexts/francais.bin", label="Français (Hugo)")

#### Texte Français 2 (Blaise Pascal)

In [ ]:
# Blaise Pascal
show_ratio("texts/Blaise_Pascal/Blaise_Pascal.txt", "texts/Blaise_Pascal/Blaise_Pascal.txt.huff", label="Blaise Pascal")

#### Texte Anglais

In [ ]:
show_ratio("texts/anglais.txt", "texts/bintexts/anglais.bin", label="Anglais")

#### Texte Grec

In [ ]:
show_ratio("texts/grec.txt", "texts/bintexts/grec.bin", label="Grec")

#### Texte Russe

In [ ]:
show_ratio("texts/russe.txt", "texts/bintexts/russe.bin", label="Russe")

#### Texte Chinois

In [ ]:
show_ratio("texts/chinois.txt", "texts/bintexts/chinois.bin", label="Chinois")

### Résultats expérimentaux sur fichiers textuels générés (question 9)
On se sert à présent de fichiers textuels qu'on génère aléatoirement grâce au script generer.sh 
(cf rapport pour explication plus détaillée de celui-ci)

In [ ]:
nb_gen = 10
for i in range(nb_gen):
    show_ratio("texts/random"+str(i)+".txt", "texts/bintexts/random"+str(i)+".huff", label="Texte généré "+str(i))

### Résultats expérimentaux sur fichiers non textuels (question 10) ###
Enfin, on fait nos dernières expérimentations en utilisant des fichiers qui contiennent
de l’information sous une autre forme : du code. 

On utilise ici des fichiers .py, .sh : on se servira tout simplement de fichiers de code présent dans ce projet.


In [ ]:
# TODO lancer la commande pour compresser les fichiers (script ?)